In [0]:
# Databricks notebook source
# =============================================================
#  GEO SAFRAS · 01_pragas_bronze_ingest.py
#  Ingestão Raw → Bronze
#  Fonte: /Volumes/workspace/pipeline_estudo/raw_files/geo_safras/
#         pragas_inseticidas_culturas.csv
#  Tabela: gs_bronze.pragas_inseticidas_culturas
#  Separador: ponto-e-vírgula (;)
# =============================================================

# COMMAND ----------
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

RAW_PATH  = "/Volumes/workspace/pipeline_estudo/raw_files/geo_safras"
ARQUIVO   = "pragas_defensivos_culturas.csv"
CATALOG   = "workspace"
SCHEMA_B  = "gs_bronze"
NOW       = datetime.now().isoformat()

print(f"[Bronze Pragas] Início: {NOW}")
print(f"[Bronze Pragas] Fonte : {RAW_PATH}/{ARQUIVO}")

# COMMAND ----------
df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .option("encoding", "utf-8")
      .option("sep", ";")
      .csv(f"{RAW_PATH}/{ARQUIVO}"))

# Remover BOM do nome da primeira coluna se existir
primeiro_col = df.columns[0]
if primeiro_col.startswith('\ufeff') or primeiro_col.startswith('ï»¿'):
    df = df.withColumnRenamed(primeiro_col, primeiro_col.lstrip('\ufeff').lstrip('ï»¿'))

# Adicionar metadados de ingestão
df = (df
      .withColumn("ingestion_timestamp", F.lit(NOW).cast("timestamp"))
      .withColumn("source_file", F.lit(ARQUIVO)))

print(f"  ✓ {ARQUIVO}: {df.count()} linhas, {len(df.columns)} colunas")

# COMMAND ----------
# MERGE Bronze — chave: cultura + praga + nivel_infestacao
if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA_B}.pragas_defensivos_culturas"):
    dt = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA_B}.pragas_defensivos_culturas")
    chave = """
        tgt.cultura          = src.cultura          AND
        tgt.praga            = src.praga            AND
        tgt.nivel_infestacao = src.nivel_infestacao
    """
    (dt.alias("tgt")
       .merge(df.alias("src"), chave)
       .whenMatchedUpdateAll()
       .whenNotMatchedInsertAll()
       .execute())
    print("  ✓ pragas_defensivos_culturas: MERGE concluído")
else:
    df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_B}.pragas_defensivos_culturas")
    print("  ✓ pragas_defensivos_culturas: criada e carregada")

# COMMAND ----------
print(f"\n[Bronze Pragas] ✅ Concluído em {datetime.now().isoformat()}")
print(f"  • {CATALOG}.{SCHEMA_B}.pragas_defensivos_culturas")